# Coleta de Classificações ATU

Este notebook tem como objetivo coletar as classificações do índice de motivos de Thompson (ATU) do site `https://www.ruthenia.ru/folklore/thompson/`.

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import string
import time
import random
import re

In [ ]:
base_url = "https://www.ruthenia.ru/folklore/thompson/"
letters = string.ascii_lowercase
data = []

for letter in letters:
    url = f"{base_url}{letter}.htm"
    print(f"Processando: {url}")
    
    try:
        response = requests.get(url)
        if response.status_code != 200:
            print(f"  Página não encontrada ou erro: {response.status_code}")
            continue
            
        # O site usa codificação windows-1251 ou similar, vamos tentar detectar ou forçar utf-8 se possível, 
        # mas o header diz charset=UTF-8 no index, vamos ver nas páginas.
        # Olhando o source anterior: <meta content="text/html; charset=UTF-8" />
        response.encoding = 'utf-8'
        
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # O conteúdo parece estar solto no body ou em tags <p>. 
        # Baseado no view_content_chunk, o texto é algo como "A0. Creator.--For a general..."
        # Vamos pegar todo o texto e tentar processar linha a linha ou parágrafo a parágrafo.
        
        # Estratégia: Iterar sobre o texto visível e usar regex para identificar códigos.
        # Padrão esperado: Letra+Número(s) + opcionalmente ponto e mais números.
        # Ex: A0., A1.1., A13.2.1.
        
        # Vamos pegar o texto cru do body para facilitar, ou iterar sobre tags 'p' se existirem.
        # O chunk mostrava texto corrido.
        
        text_content = soup.get_text(separator='\n')
        lines = text_content.split('\n')
        
        current_code = None
        current_desc = []
        
        # Regex para capturar o código no início da linha.
        # Ex: A0. Creator.
        # Às vezes aparece duplicado: "A0. A0. Creator."
        code_pattern = re.compile(r'^\s*([A-Z][0-9]+(?:\.[0-9]+)*)\.?\s+(.*)')
        
        for line in lines:
            line = line.strip()
            if not line:
                continue
                
            match = code_pattern.match(line)
            if match:
                # Se já tínhamos um código sendo processado, salva ele (embora aqui estamos processando linha a linha)
                # Mas a descrição pode continuar na próxima linha? 
                # Pelo chunk: "A0. A0. Creator.--For a general..."
                # Parece que é tudo numa linha ou parágrafo.
                
                code = match.group(1)
                desc = match.group(2)
                
                # Limpeza da descrição
                # Remove repetição do código se houver (ex: "A0. Creator" -> desc="A0. Creator")
                # O regex pega o primeiro código. O resto fica no grupo 2.
                # Se o grupo 2 começar com o mesmo código, removemos.
                if desc.startswith(code):
                    desc = desc[len(code):].strip(" .-")
                
                # Limpa caracteres estranhos do início
                desc = desc.lstrip(".- ")
                
                data.append({
                    'code': code,
                    'description': desc,
                    'source_url': url
                })
                
    except Exception as e:
        print(f"Erro ao processar {url}: {e}")
        
    # Pausa aleatória para não sobrecarregar o servidor (entre 1 e 3 segundos)
    sleep_time = random.uniform(1, 3)
    time.sleep(sleep_time)

df = pd.DataFrame(data)
print(f"Total de registros coletados: {len(df)}")

Processando: https://www.ruthenia.ru/folklore/thompson/a.htm
Processando: https://www.ruthenia.ru/folklore/thompson/b.htm
Processando: https://www.ruthenia.ru/folklore/thompson/c.htm
Processando: https://www.ruthenia.ru/folklore/thompson/d.htm
Processando: https://www.ruthenia.ru/folklore/thompson/e.htm
Processando: https://www.ruthenia.ru/folklore/thompson/f.htm
Processando: https://www.ruthenia.ru/folklore/thompson/g.htm
Processando: https://www.ruthenia.ru/folklore/thompson/h.htm
Processando: https://www.ruthenia.ru/folklore/thompson/i.htm
  Página não encontrada ou erro: 404
Processando: https://www.ruthenia.ru/folklore/thompson/j.htm
Processando: https://www.ruthenia.ru/folklore/thompson/k.htm
Processando: https://www.ruthenia.ru/folklore/thompson/l.htm
Processando: https://www.ruthenia.ru/folklore/thompson/m.htm
Processando: https://www.ruthenia.ru/folklore/thompson/n.htm
Processando: https://www.ruthenia.ru/folklore/thompson/o.htm
  Página não encontrada ou erro: 404
Processando

,code,description,source_url
0,A0,Creator.,https://www.ruthenia.ru/folklore/thompson/a.htm
1,A10,Nature of the creator.,https://www.ruthenia.ru/folklore/thompson/a.htm
2,A20,Origin of the creator.,https://www.ruthenia.ru/folklore/thompson/a.htm
3,A30,Creator,https://www.ruthenia.ru/folklore/thompson/a.htm
4,A40,Creator,https://www.ruthenia.ru/folklore/thompson/a.htm


In [41]:
# df.rename(columns={'code': 'code_atu'}, inplace=True)
df= df.dropna(subset=['description'])
df = df[df['description'].notna() & (df['description'].str.strip() != '')]
df.sample(10)

,code_atu,description,source_url
27763,J2210,Logical absurdity,https://www.ruthenia.ru/folklore/thompson/j.htm
35645,M150,Other vows and,https://www.ruthenia.ru/folklore/thompson/m.htm
667,A1020,Escape from,https://www.ruthenia.ru/folklore/thompson/a.htm
20050,F480,House-spirits.,https://www.ruthenia.ru/folklore/thompson/f.htm
11266,D1350,Magic object,https://www.ruthenia.ru/folklore/thompson/d.htm
42373,T570,Pregnancy.,https://www.ruthenia.ru/folklore/thompson/t.htm
1573,A1520,Origin of hunting,https://www.ruthenia.ru/folklore/thompson/a.htm
6834,C330,Tabu: looking in,https://www.ruthenia.ru/folklore/thompson/c.htm
1976,A1860,Creation of,https://www.ruthenia.ru/folklore/thompson/a.htm
17702,F760,Extraordinary,https://www.ruthenia.ru/folklore/thompson/f.htm


In [ ]:
# # Salvar em CSV
# df.to_csv('atu_classifications.csv', index=False)
# print("Arquivo 'atu_classifications.csv' salvo com sucesso.")

Arquivo 'atu_classifications.csv' salvo com sucesso.


In [3]:
import pandas as pd
import sqlite3

# caminho para o banco de dados
db_path = '../data/contos.sqlite'
conn = sqlite3.connect(db_path)

df_tales = pd.read_sql_query('SELECT * FROM tales', conn)

In [4]:
def extrair_classificacao(url):
    # Verifica se a URL é válida (não é nula)
    if not isinstance(url, str):
        return pd.Series([None, None])
    
    try:
        # Lógica de extração original, mas mais segura com split
        # Pega o que está depois de 'dash/' e antes de '.html'
        slug = url.split('dash/')[-1].split('.html')[0]
        
        if 'type' in slug:
            # Se tem 'type', é ATU e removemos o prefixo 'type' (4 caracteres)
            return pd.Series(['ATU', slug.replace('type', '')])
        else:
            # Caso contrário, é outra classificação e mantemos o slug original
            return pd.Series(['Other classification', slug])
            
    except IndexError:
        # Caso a URL não siga o padrão esperado
        return pd.Series([None, None])

# Cria as duas novas colunas de uma vez
df_tales[['tipo_classificacao', 'codigo_classificacao']] = df_tales['url'].apply(extrair_classificacao)

# Visualiza o resultado
df_tales[['url', 'tipo_classificacao', 'codigo_classificacao']].sample(5)

,url,tipo_classificacao,codigo_classificacao
185,https://sites.pitt.edu/~dash/type1626.html,ATU,1626
589,https://sites.pitt.edu/~dash/hand.html,Other classification,hand
1541,https://sites.pitt.edu/~dash/aesopold.html,Other classification,aesopold
833,https://sites.pitt.edu/~dash/bogey.html,Other classification,bogey
572,https://sites.pitt.edu/~dash/cain.html,Other classification,cain


In [6]:
df_tales.sample(3)

,id,titulo,origem,url,texto_completo,tipo_classificacao,codigo_classificacao
837,878,The Hard-Hearted Father,Austria,https://sites.pitt.edu/~dash/bogey.html,His father was an evil-tempered laborer who wa...,Other classification,bogey
690,723,How Eulenspiegel Painted the Forbears of the L...,Germany,https://sites.pitt.edu/~dash/type1620.html,"Then inquired the landgrave of Eulenspiegel, w...",ATU,1620
975,1021,The Origin of theTusser,Norway,https://sites.pitt.edu/~dash/originunder.html,"""Yes,"" said the man. ""And if you will wait her...",Other classification,originunder


In [19]:
df_unc_clssif = df_tales[['tipo_classificacao', 'codigo_classificacao']].drop_duplicates().sort_values(by=['tipo_classificacao', 'codigo_classificacao'])
df_unc_clssif.rename(columns={'tipo_classificacao': 'classification_system', 'codigo_classificacao': 'classification_name'}, inplace=True)
df_unc_clssif.drop_duplicates(subset=['classification_name'], inplace=True)
df_unc_clssif.sample(5)

df_unc_clssif.to_csv('classificacoes_unicas_atu_other.csv', index=False)


In [ ]:
# Renomeia a coluna code_atu para facilitar o merge
df_atu = df.rename(columns={'code_atu': 'codigo_classificacao'})

# Faz o merge apenas para registros ATU
df_tales = df_tales.merge(
    df_atu[['codigo_classificacao', 'description']], 
    on='codigo_classificacao', 
    how='left'
)

# Renomeia a coluna description para atu_description
df_tales = df_tales.rename(columns={'description': 'atu_description'})

# Limpa atu_description para registros que não são ATU
df_tales.loc[df_tales['tipo_classificacao'] != 'ATU', 'atu_description'] = None

# Visualiza o resultado
df_tales[df_tales['tipo_classificacao'] == 'ATU'][['codigo_classificacao', 'atu_description']].sample(5)

NameError: name 'df' is not defined